# Trial Demo
Run a trial end-to-end: create a trial, create a fixed route, record frames + risk, optionally append model risk, and generate a video with a risk curve tile.

In [1]:
# Imports
import os
import time
from pathlib import Path

from MIREIA.config import Config
from MIREIA.simulation.world_manager import WorldManager

In [2]:
# Launch CARLA (skip if already running)
import subprocess
subprocess.Popen("CarlaUE4.exe", shell=True)

<Popen: returncode: None args: 'CarlaUE4.exe'>

In [8]:
import random
import carla
from MIREIA.config import Config
from MIREIA.simulation.traffic_handler import TrafficHandler
from MIREIA.simulation.simple_route_controller import SimpleRouteController

# reload packages (for development)
import importlib

# Direct run: spawn ego + attach controller (no Scenario, no WorldManager)
client = carla.Client(Config.CARLA_HOST, Config.CARLA_PORT)
client.set_timeout(15.0)
world = client.get_world()
world_map = world.get_map()

# Keep previous world settings to restore later
prev_settings = world.get_settings()
settings = world.get_settings()
settings.synchronous_mode = True
settings.fixed_delta_seconds = 0.05
world.apply_settings(settings)

spawn_points = world_map.get_spawn_points()
if len(spawn_points) < 2:
    raise RuntimeError("Need at least 2 spawn points to run a route.")

# Randomize ego spawn position each execution
ego_spawn_index = random.randrange(len(spawn_points))

handler = TrafficHandler(client, world, seed=42)
controller = SimpleRouteController(target_speed=10.0, sampling_resolution=2.0)

ego = handler.spawn_ego(
    blueprint_id="vehicle.lincoln.mkz_2020",
    spawn_index=ego_spawn_index,
    autopilot=False,
    controller=controller,
)

if hasattr(carla, "VehicleLightState"):
    light_state = (
        carla.VehicleLightState.Position
        | carla.VehicleLightState.LowBeam
        | carla.VehicleLightState.Interior
    )
    ego.set_light_state(carla.VehicleLightState(light_state))

# Let CARLA settle one frame so ego transform and waypoint are consistent
world.tick()

start = ego.get_location()

# Build valid destination candidates, then choose one randomly each execution
candidates = []
for i, sp in enumerate(spawn_points):
    if i == ego_spawn_index:
        continue
    d = start.distance(sp.location)
    if d > 35.0:
        candidates.append((d, i))

if not candidates:
    raise RuntimeError("No suitable destination spawn point found.")

_, best_idx = random.choice(candidates)
end = spawn_points[best_idx].location
controller.set_destination(start, end)
print(f"Plan length: {controller.get_plan_length()} waypoints")

print(f"Spawn index: {ego_spawn_index}")
print(f"Start: ({start.x:.1f}, {start.y:.1f})")
print(f"End:   ({end.x:.1f}, {end.y:.1f})  [spawn index {best_idx}]")

def draw_ego_glow(vehicle: carla.Vehicle):
    t = vehicle.get_transform()
    loc = t.location
    extent = vehicle.bounding_box.extent
    world.debug.draw_box(
        box=carla.BoundingBox(loc, extent),
        rotation=t.rotation,
        thickness=0.15,
        color=carla.Color(0, 255, 255),
        life_time=0.08,
    )
    world.debug.draw_point(
        location=carla.Location(x=loc.x, y=loc.y, z=loc.z + 1.0),
        size=0.2,
        color=carla.Color(255, 255, 0),
        life_time=0.08,
    )

for step in range(1000):
    handler.run_ego_controller_step()
    world.tick()
    draw_ego_glow(ego)
    controller.draw_plan(world, max_points=1400, life_time=0.08)
    if controller.done():
        print(f"Route finished at step {step}")
        for i in range(100):
            draw_ego_glow(ego)
            world.tick()
        break


handler.destroy_all()
world.apply_settings(prev_settings)
print("Done.")

Spawned ego vehicle 'vehicle.lincoln.mkz_2020' at index 90 (autopilot=False)
Plan length: 145 waypoints
Spawn index: 90
Start: (-64.1, 16.5)
End:   (17.1, 28.1)  [spawn index 57]
Destroyed ego vehicle.
Done.


## 1) Create or load a Trial
A Trial owns its folder under `PATH_TO_TRIALS` and references a `route.json` that contains waypoint IDs.

In [6]:
trial_name = "demo_trial_001"
trial = Trial(
    name=trial_name,
    map_name="Town03",
    weather="ClearNoon",
    description="Demo trial with predefined route.",
    n_vehicles=0,
    n_pedestrians=0,
 )
trial.save()
print(f"Trial saved at: {trial.folder_path}")
print(f"Route JSON: {trial.route_path}")

Trial saved at: d:\Projectes\TFG\MIREIA\trials\demo_trial_001
Route JSON: d:\Projectes\TFG\MIREIA\trials\demo_trial_001\route.json


## 2) Create a route (run once)
Use the interactive picker to select waypoints and save `route.json`. Skip this if the route already exists.

wm = WorldManager(verbose=True)
wm.load_scenario(trial)

route = create_route_from_waypoints(wm.bridge.get_waypoints())
save_route(route, trial.route_path)
plot_route(wm.bridge.get_waypoints(), [wp.id for wp in route.waypoints])

wm.destroy()
print(f"Route saved to: {trial.route_path}")

In [7]:
run_id, run_path = trial.create_run_folder()
run_path = Path(run_path)
dataset_jsonl = run_path / "dataset.jsonl"
images_dir = run_path / "images"
images_dir.mkdir(parents=True, exist_ok=True)

static_meta = {
    "trial_name": trial.name,
    "run_id": run_id,
}
print(f"Run folder: {run_path}")
print(f"JSONL path: {dataset_jsonl}")

Run folder: d:\Projectes\TFG\MIREIA\trials\demo_trial_001\runs\20260323_221836
JSONL path: d:\Projectes\TFG\MIREIA\trials\demo_trial_001\runs\20260323_221836\dataset.jsonl


## 4) Run a trial
The current flow uses autopilot only. A route-following controller can be added later.

In [8]:
use_controller = False  # False = autopilot
compute_model_risk = False
num_frames = 200
frame_stride = 5

wm = WorldManager(verbose=True)
wm.load_scenario(trial)
wm.enable_recording(
    append=False,
    include_topdown=False,
    include_static_risk_image=False,
    jsonl_path=str(dataset_jsonl),
    static_meta=static_meta,
 )
wm.setup_sensors(save_dir=str(images_dir), enable_map_camera=False)
delta_seconds = wm.world.get_settings().fixed_delta_seconds or 0.05

start_time = time.time()
for frame_id in range(num_frames):
    if frame_id % frame_stride != 0:
        wm.tick()
        continue

    rgb_path = images_dir / f"rgb_{frame_id:06d}.png"
    rel_rgb_path = str(rgb_path.relative_to(run_path))

    def tick_and_log():
        ctrl = wm.ego_vehicle.get_control() if wm.ego_vehicle is not None else None
        extra_fields = {
            "control": {
                "throttle": ctrl.throttle if ctrl else 0.0,
                "steer": ctrl.steer if ctrl else 0.0,
                "brake": ctrl.brake if ctrl else 0.0,
            },
            "mode": "controller" if use_controller else "autopilot",
        }
        wm.tick(
            ground_truth_risk=None,
            rgb_image_path=rel_rgb_path,
            extra_fields=extra_fields,
        )

    wm.sensor_manager.save_ego_frame(save_path=str(rgb_path), tick_fn=tick_and_log)

end_time = time.time()
elapsed = end_time - start_time
trial.add_run_summary(run_id, {"elapsed_seconds": elapsed, "frames": num_frames, "delta_seconds": delta_seconds})
print(f"Trial run completed in {elapsed:.1f}s")

wm.destroy()

Connecting to CARLA at 127.0.0.1:2000...
Connected. Current map: 'Carla/Maps/Town10HD_Opt'.
Loading scenario 'demo_trial_001' (map=Town03, seed=42)...
Switching map from 'Carla/Maps/Town10HD_Opt' to 'Town03'...
Weather set to 'ClearNoon'.
Spawned ego vehicle 'vehicle.lincoln.mkz_2020' at index None (autopilot=True)
SimulationBridge initialized: SimulationBridge(Ego: None, Dynamic Obstacles: 0, Static Obstacles: 7092, EnvState: (Visibility: 282.0m, Friction: 0.80))
Computing static risk map with bounds: (49.96485900878906, 0.818756103515625, 456.9049987792969) and resolution: 2.0m...
Scenario 'demo_trial_001' is ready.
Recording enabled → d:\Projectes\TFG\MIREIA\trials\demo_trial_001\runs\20260323_221836\dataset.jsonl


AttributeError: 'NoneType' object has no attribute 'x'

## 5) Append model risk (optional)
Run post-inference to add `model_risk` per frame. Skip for autopilot-only runs.

In [ ]:
if compute_model_risk:
    import json
    import torch
    from PIL import Image

    from MIREIA.data_collection.dataset_utils import (
        build_default_transform,
        load_jsonl_records,
        resolve_image_path,
    )
    from MIREIA.perception.e2e_model import E2EModelConfig, E2ERiskPredictor

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    transform = build_default_transform()

    def load_checkpoint(model: torch.nn.Module, checkpoint_path: str) -> None:
        state = torch.load(checkpoint_path, map_location=device)
        if isinstance(state, dict):
            if "model_state_dict" in state:
                state = state["model_state_dict"]
            elif "model_state" in state:
                state = state["model_state"]
            elif "state_dict" in state:
                state = state["state_dict"]
        model.load_state_dict(state)
        model.eval()

    def load_image(path: str) -> torch.Tensor:
        with Image.open(path) as img:
            img = img.convert("RGB")
            return transform(img)

    def predict_from_paths(model: torch.nn.Module, image_paths: list[str]) -> float:
        seq_tensor = torch.stack([load_image(path) for path in image_paths], dim=0)
        if seq_tensor.ndim == 4:
            seq_tensor = seq_tensor.unsqueeze(0)
        seq_tensor = seq_tensor.to(device)
        with torch.no_grad():
            pred = model(seq_tensor)
        return float(pred.squeeze().item())

    def append_model_risk_to_jsonl(
        jsonl_path: str,
        model: torch.nn.Module,
        seq_len: int,
        output_key: str = "model_risk",
        image_root: str | None = None,
        input_key: str = "rgb_image_path",
        normalize_paths: bool = True,
    ) -> None:
        if seq_len <= 0:
            raise ValueError("seq_len must be > 0")
        records = load_jsonl_records(jsonl_path)
        if not records:
            raise RuntimeError(f"No records found in {jsonl_path}")
        base_dir = image_root or os.path.dirname(jsonl_path)
        for idx, record in enumerate(records):
            if idx + 1 < seq_len:
                record[output_key] = None
                continue
            window = records[idx - seq_len + 1 : idx + 1]
            image_paths: list[str] = []
            for rec in window:
                rel_path = rec.get(input_key, "")
                if not rel_path:
                    raise ValueError(f"Missing {input_key} in {jsonl_path}")
                image_paths.append(resolve_image_path(base_dir, rel_path, normalize_paths))
            record[output_key] = predict_from_paths(model, image_paths)
        with open(jsonl_path, "w", encoding="utf-8") as handle:
            for record in records:
                handle.write(json.dumps(record) + "\n")

    checkpoint_path = os.path.join(Config.PATH_TO_MODELS, "single_checkpoint.pt")
    model = E2ERiskPredictor(E2EModelConfig()).to(device)
    load_checkpoint(model, checkpoint_path)
    append_model_risk_to_jsonl(
        jsonl_path=str(dataset_jsonl),
        model=model,
        seq_len=5,
        image_root=str(run_path),
    )
    print("Model risk appended to JSONL")
else:
    print("Skipping model risk")

## 6) Compose the video with risk chart
This renders a line chart tile with ground-truth risk and prediction if present.

In [ ]:
from MIREIA.analysis.visualizer import DatasetVideoComposer

composer = DatasetVideoComposer(
    str(dataset_jsonl),
    fps=10,
    risk_tile_mode="curve",
    delta_seconds=delta_seconds,
 )
video_path = composer.build_video(output_name="trial_video.mp4")
print(f"Saved video: {video_path}")

## 7) Compare two runs (overlay + AUC)
Set two JSONL paths from different runs to generate an overlay plot.

In [ ]:
# Example: compare the latest run with a previous run.
# Update these paths to match your runs.
run_a_jsonl = str(dataset_jsonl)
run_b_jsonl = ""  # e.g., r"D:/Projectes/TFG/MIREIA/trials/demo_trial_001/runs/20260322_214235/dataset.jsonl"

if run_b_jsonl:
    aucs = plot_risk_curves(
        [run_a_jsonl, run_b_jsonl],
        labels=["run_a", "run_b"],
        output_path=str(run_path / "risk_compare.png"),
        key="ground_truth_risk",
        delta_seconds=delta_seconds,
    )
    print("Saved overlay plot to", run_path / "risk_compare.png")
    print("AUCs:", aucs)
else:
    print("Set run_b_jsonl to compare runs")